In [ ]:
import sys
from dotenv import load_dotenv

sys.path.insert(0, '.')
load_dotenv(override=True)

from src.data.fred import (
    fetch_treasury_rates,
    fetch_gdp_growth,
    fetch_fed_funds,
    fetch_unemployment,
    fetch_cpi,
    fetch_hy_oas,
)
print('imports ok')

imports ok


In [11]:
try:
    df = fetch_treasury_rates()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(6522, 3)  range=2000-01-03 -> 2024-12-31


,treasury_3m,treasury_10y,term_spread
date,,,
2000-01-03,5.48,6.58,1.10
2000-01-04,5.43,6.49,1.06
2000-01-05,5.44,6.62,1.18


In [12]:
try:
    df = fetch_gdp_growth()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(100, 1)  range=2000-01-01 -> 2024-10-01


,gdp_growth_qoq
date,
2000-01-01,1.5
2000-04-01,7.5
2000-07-01,0.4


In [13]:
try:
    df = fetch_fed_funds()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(300, 1)  range=2000-01-01 -> 2024-12-01


,fed_funds_rate
date,
2000-01-01,5.45
2000-02-01,5.73
2000-03-01,5.85


In [14]:
try:
    df = fetch_unemployment()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(300, 2)  range=2000-01-01 -> 2024-12-01


,unemployment_rate,unemployment_change
date,,
2000-01-01,4.0,NaN
2000-02-01,4.1,0.1
2000-03-01,4.0,-0.1


In [15]:
try:
    df = fetch_cpi()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(300, 2)  range=2000-01-01 -> 2024-12-01


,cpi,cpi_yoy
date,,
2000-01-01,169.3,NaN
2000-02-01,170.0,NaN
2000-03-01,171.0,NaN


In [16]:
try:
    df = fetch_hy_oas()
    print(f"PASS  shape={df.shape}  range={df.index.min().date()} -> {df.index.max().date()}")
    display(df.head(3))
except Exception as e:
    print(f"FAIL  {e}")

PASS  shape=(414, 1)  range=2023-06-09 -> 2024-12-31


,hy_oas
date,
2023-06-09,4.29
2023-06-12,4.34
2023-06-13,4.18


In [1]:
from src.data.wrds import WRDSClient

db = WRDSClient()  

Loading library list...
Done


In [10]:
# 1. Flattened WRDS key developments table
a = db.query("""
SELECT DISTINCT eventtype
FROM ciq.wrds_keydev
ORDER BY eventtype
""")

In [ ]:
display(db.query("""
SELECT eventtype,
       COUNT(*) AS n_events,
       COUNT(DISTINCT companyid) AS n_companies,
       COUNT(DISTINCT gvkey) AS n_gvkeys
FROM ciq.wrds_keydev
WHERE eventtype IN (
    'Debt Defaults',
    'Bankruptcy - Filing',
    'Bankruptcy - Reorganization',
    'Bankruptcy - Asset Sale/Liquidation',
    'Bankruptcy - Emergence/Exit',
    'Bankruptcy - Conclusion'
)
AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
GROUP BY eventtype
ORDER BY n_events DESC
"""))

   announcedate   gvkey                                              companyname            eventtype                                                                                                                                             headline
0    2026-06-12    <NA>                          Sleep Number Health Corporation  Bankruptcy - Filing                                                                                                        Sleep Number Corporation Filed for Bankruptcy
1    2026-06-12    <NA>                                                     <NA>  Bankruptcy - Filing                                                                                                        Sleep Number Corporation Filed for Bankruptcy
2    2026-06-12    <NA>                        Select Comfort Canada Holding Inc  Bankruptcy - Filing                                                                                                        Sleep Number Corporation Filed for Bankrup

In [ ]:
print(a.to_string())

In [17]:
display(db.query("""
SELECT eventtype,
       COUNT(*) AS n_events,
       COUNT(DISTINCT companyid) AS n_companies,
       COUNT(DISTINCT gvkey) AS n_gvkeys
FROM ciq.wrds_keydev
WHERE eventtype IN (
    'Debt Defaults',
    'Bankruptcy - Filing',
    'Bankruptcy - Reorganization',
    'Bankruptcy - Asset Sale/Liquidation',
    'Bankruptcy - Emergence/Exit',
    'Bankruptcy - Conclusion'
)
AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
GROUP BY eventtype
ORDER BY n_events DESC
"""))

,eventtype,n_events,n_companies,n_gvkeys
0,Bankruptcy - Filing,117645,83117,3148
1,Bankruptcy - Conclusion,35765,32654,1332
2,Bankruptcy - Emergence/Exit,32388,26944,1605
3,Debt Defaults,2653,1384,821


In [20]:
print(db.list_tables('dealscan'))

['borrowerbase', 'chars', 'company', 'currfacpricing', 'dealamendment', 'dealpurposecomment', 'dealscan', 'facility', 'facilityamendment', 'facilitydates', 'facilityguarantor', 'facilitypaymentschedule', 'facilityrepaymentcomment', 'facilitysecurity', 'facilitysponsor', 'financialcovenant', 'financialratios', 'lendershares', 'lins', 'lpc_loanconnector_company_id_map', 'marketsegment', 'networthcovenant', 'organizationtype', 'package', 'packageassignmentcomment', 'packageprepaymentcomment', 'performancepricing', 'performancepricingcomments', 'sublimits', 'wrds_financial_covenants', 'wrds_loanconnector_ids']


In [21]:
display(db.query("""
SELECT *
FROM dealscan.wrds_loanconnector_ids
LIMIT 20
"""))

,loanconnector_deal_id,wrds_package_id,loanconnector_tranche_id,wrds_facility_id
0,176568,100.0,261534,1.0
1,176568,100.0,290057,2.0
2,176568,100.0,208427,3.0
3,176568,100.0,208047,4.0
4,144226,101.0,235669,5.0
5,29400,102.0,64192,6.0
6,29400,102.0,64193,7.0
7,29400,102.0,69385,8.0
8,96297,103.0,143042,9.0
9,161210,104.0,251261,10.0


In [22]:
display(db.query("""
SELECT *
FROM dealscan.lpc_loanconnector_company_id_map
LIMIT 20
"""))

,company_name,loanconnector_company_id,lpc_company_id
0,IBC Acquisitions,73250,1.0
1,UIS Inc,65640,2.0
2,Chatswood Inc,72394,3.0
3,American Community Development Group,62957,4.0
4,Hoker Broadcasting,78489,6.0
5,MHI Group,69533,7.0
6,Plastics Manufacturing Co,70248,8.0
7,Plum Associates,76649,9.0
8,Southlife Holding Co,71449,10.0
9,TCA-IV LP,81132,11.0


In [23]:
display(db.query("""
SELECT *
FROM dealscan.company
LIMIT 20
"""))

,companyid,company,parentid,ultimateparentid,sales,ticker,publicprivate,city,state,country,zipcode,region,institutiontype,primarysiccode,secondarysiccode,tertiarysiccode
0,1.0,IBC Acquisitions,<NA>,1.0,<NA>,<NA>,Private,Boston,Massachusetts,USA,<NA>,North America,<NA>,6799,<NA>,<NA>
1,2.0,UIS Inc,<NA>,2.0,<NA>,<NA>,Private,New York,New York,USA,<NA>,North America,<NA>,3714,2064,3322
2,3.0,Chatswood Inc,<NA>,3.0,<NA>,<NA>,Private,San Francisco,California,USA,<NA>,North America,<NA>,6799,<NA>,<NA>
3,4.0,American Community Development Group,<NA>,4.0,<NA>,<NA>,Private,Petersburg,Florida,USA,<NA>,North America,<NA>,6553,<NA>,<NA>
4,6.0,Hoker Broadcasting,<NA>,6.0,<NA>,<NA>,Private,Dallas,Texas,USA,<NA>,North America,<NA>,4832,<NA>,<NA>
5,7.0,MHI Group,<NA>,7.0,<NA>,MH,Public,Tallahassee,Florida,USA,<NA>,North America,<NA>,7261,<NA>,<NA>
6,8.0,Plastics Manufacturing Co,<NA>,8.0,<NA>,<NA>,Private,Dallas,Texas,USA,<NA>,North America,<NA>,3089,2821,<NA>
7,9.0,Plum Associates,<NA>,9.0,<NA>,<NA>,Private,New York,New York,USA,<NA>,North America,<NA>,5141,<NA>,<NA>
8,10.0,Southlife Holding Co,<NA>,10.0,<NA>,SLHC,Public,Nashville,Tennessee,USA,<NA>,North America,<NA>,<NA>,<NA>,<NA>
9,11.0,TCA-IV LP,<NA>,11.0,<NA>,<NA>,Private,New Haven,Connecticut,USA,<NA>,North America,<NA>,6719,<NA>,<NA>


In [24]:
display(db.query("""
SELECT m.company_name,
       m.loanconnector_company_id,
       m.lpc_company_id,
       c.companyid,
       c.company,
       c.ticker,
       c.publicprivate,
       c.country
FROM dealscan.lpc_loanconnector_company_id_map m
LEFT JOIN dealscan.company c
  ON m.lpc_company_id = c.companyid
LIMIT 20
"""))

,company_name,loanconnector_company_id,lpc_company_id,companyid,company,ticker,publicprivate,country
0,IBC Acquisitions,73250,1.0,1.0,IBC Acquisitions,<NA>,Private,USA
1,UIS Inc,65640,2.0,2.0,UIS Inc,<NA>,Private,USA
2,Chatswood Inc,72394,3.0,3.0,Chatswood Inc,<NA>,Private,USA
3,American Community Development Group,62957,4.0,4.0,American Community Development Group,<NA>,Private,USA
4,Hoker Broadcasting,78489,6.0,6.0,Hoker Broadcasting,<NA>,Private,USA
5,MHI Group,69533,7.0,7.0,MHI Group,MH,Public,USA
6,Plastics Manufacturing Co,70248,8.0,8.0,Plastics Manufacturing Co,<NA>,Private,USA
7,Plum Associates,76649,9.0,9.0,Plum Associates,<NA>,Private,USA
8,Southlife Holding Co,71449,10.0,10.0,Southlife Holding Co,SLHC,Public,USA
9,TCA-IV LP,81132,11.0,11.0,TCA-IV LP,<NA>,Private,USA


In [26]:
display(db.query("""
SELECT *
FROM dealscan.package
LIMIT 20
"""))

,packageid,borrowercompanyid,ticker,dealactivedate,company,comment,dealamount,currency,exchangerate,salesatclose,...,companyconsent,defaultbaserate,spreadoverdefaultbase,prorataallocation,assetsalessweep,debtissuancesweep,equityissuancesweep,dividendrestrictions,insuranceproceedssweep,requiredlenders
0,100.0,33181.0,<NA>,1987-09-12,Sager Electical Supply Co,<NA>,50000000.0,United States Dollars,1.0,77600000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,101.0,3769.0,SGAL,1987-08-05,Sage-Allen Co,<NA>,18000000.0,United States Dollars,1.0,77600000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,102.0,1.0,<NA>,1987-09-12,IBC Acquisitions,<NA>,370000000.0,United States Dollars,1.0,749337000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,103.0,2.0,<NA>,1987-09-01,UIS Inc,<NA>,10000000.0,United States Dollars,1.0,469482000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,104.0,2.0,<NA>,1987-09-08,UIS Inc,<NA>,10000000.0,United States Dollars,1.0,469482000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,105.0,3782.0,<NA>,1987-09-01,Hanson Trust,<NA>,2414500000.0,United States Dollars,1.0,<NA>,...,<NA>,<NA>,<NA>,Yes,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,106.0,3.0,<NA>,1987-10-01,Chatswood Inc,<NA>,10000000.0,United States Dollars,1.0,54741000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,107.0,4.0,<NA>,1987-08-01,American Community Development Group,<NA>,10000000.0,United States Dollars,1.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,108.0,4.0,<NA>,1987-08-01,American Community Development Group,<NA>,15400000.0,United States Dollars,1.0,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,109.0,3803.0,AMHD,1987-01-12,American Museum,<NA>,600000.0,United States Dollars,1.0,1345000.0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [27]:
display(db.query("""
SELECT *
FROM dealscan.facility
LIMIT 20
"""))

,facilityid,packageid,borrowercompanyid,ticker,facilitystartdate,facilityenddate,company,comment,targetcompany,loantype,...,exchangerate,maturity,secured,lclimit,renewal,distributionmethod,averagelife,seniority,countryofsyndication,conversiondate
0,1.0,100.0,33181.0,<NA>,1987-09-12,1995-09-01,Sager Electical Supply Co,The 45 share warrants must be purchased and it...,<NA>,Term Loan,...,1.0,96,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
1,2.0,100.0,33181.0,<NA>,1987-09-12,1989-09-01,Sager Electical Supply Co,Spread rises to 175 for 13-18 months and to 20...,<NA>,Term Loan,...,1.0,24,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
2,3.0,100.0,33181.0,<NA>,1987-09-12,1995-09-01,Sager Electical Supply Co,Secured by pledge of all capital stock.,<NA>,Term Loan,...,1.0,96,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
3,4.0,100.0,33181.0,<NA>,1987-09-12,1995-09-01,Sager Electical Supply Co,5% prepayment penalty.,<NA>,Revolver/Line >= 1 Yr.,...,1.0,96,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
4,5.0,101.0,3769.0,SGAL,1987-08-05,1987-11-05,Sage-Allen Co,Spread rises to 650 after maturity.,<NA>,Bridge Loan,...,1.0,3,Yes,<NA>,<NA>,Syndication,<NA>,Senior,USA,<NA>
5,6.0,102.0,1.0,<NA>,1987-09-12,1992-11-01,IBC Acquisitions,TorDom can participate down commit. to $40 Mil...,<NA>,Limited Line,...,1.0,62,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
6,7.0,102.0,1.0,<NA>,1987-09-12,1995-11-01,IBC Acquisitions,Step-downs in pricing to be based on ratio of ...,<NA>,Limited Line,...,1.0,98,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
7,8.0,102.0,1.0,<NA>,1987-09-12,1988-01-12,IBC Acquisitions,25 bps of upfront fees is on total commit. fro...,Interstate Bakeries Corp,Term Loan,...,1.0,4,Yes,<NA>,<NA>,<NA>,<NA>,Senior,USA,<NA>
8,9.0,103.0,2.0,<NA>,1987-09-01,1988-09-01,UIS Inc,<NA>,Walbro Corp,Limited Line,...,1.0,12,No,<NA>,<NA>,Sole Lender,<NA>,Senior,USA,<NA>
9,10.0,104.0,2.0,<NA>,1987-09-08,1988-09-08,UIS Inc,Credit was increased from a $3M Line of Credit...,<NA>,Limited Line,...,1.0,12,No,<NA>,<NA>,Sole Lender,<NA>,Senior,USA,<NA>


In [36]:
cols = db.query("""
SELECT *
FROM dealscan.dealscan
LIMIT 1
""").columns

for c in cols:
    if any(x in c.lower() for x in ['borrower', 'company', 'gvkey', 'cusip', 'ticker', 'facility', 'package', 'loanconnector']):
        print(c)

borrower_name
borrower_id
additional_borrowers
ticker
borrower_type
parent_ticker
company_url
tranche_cusip
borrower_consent
law_firm_borrower_primary
law_firm_borrower_other


In [33]:
db._db.connection.rollback()

In [39]:
display(db.query("""
SELECT f.facilityid,
       f.packageid,
       f.borrowercompanyid,
       p.company AS package_company,
       c.company AS company_table_name,
       c.ticker,
       c.publicprivate,
       c.country
FROM dealscan.facility f
LEFT JOIN dealscan.package p
  ON f.packageid = p.packageid
LEFT JOIN dealscan.company c
  ON f.borrowercompanyid = c.companyid
LIMIT 20
"""))

,facilityid,packageid,borrowercompanyid,package_company,company_table_name,ticker,publicprivate,country
0,1.0,100.0,33181.0,Sager Electical Supply Co,Sager Electical Supply Co,<NA>,<NA>,USA
1,2.0,100.0,33181.0,Sager Electical Supply Co,Sager Electical Supply Co,<NA>,<NA>,USA
2,3.0,100.0,33181.0,Sager Electical Supply Co,Sager Electical Supply Co,<NA>,<NA>,USA
3,4.0,100.0,33181.0,Sager Electical Supply Co,Sager Electical Supply Co,<NA>,<NA>,USA
4,5.0,101.0,3769.0,Sage-Allen Co,Sage-Allen Co,SGAL,Public,USA
5,6.0,102.0,1.0,IBC Acquisitions,IBC Acquisitions,<NA>,Private,USA
6,7.0,102.0,1.0,IBC Acquisitions,IBC Acquisitions,<NA>,Private,USA
7,8.0,102.0,1.0,IBC Acquisitions,IBC Acquisitions,<NA>,Private,USA
8,9.0,103.0,2.0,UIS Inc,UIS Inc,<NA>,Private,USA
9,10.0,104.0,2.0,UIS Inc,UIS Inc,<NA>,Private,USA


In [42]:
display(db.query("""
SELECT *
FROM wrdsapps_link_dealscan_wscope.dswslink
LIMIT 20
"""))

,companyid,company,cleaned_matched_name,code,cusip,sedol,isin,ibtic,iso3,sic,spedis_ratio,sic3dum,seciddum,geodum,iso3dum,fdate
0,4.0,AMERICAN COMMUNITY DEVELOPMENT GROUP,AMERICAN CMNTY DEVLOPMENT GP,<NA>,02520G101,<NA>,<NA>,CHZM,USA,6553.0,88.0,0.0,0.0,1.0,1.0,2020-10-20
1,17.0,JARDINE STRATEGIC HOLDINGS LTD,JARDINE STRATEGIC HOLDINGS LTD,10897.0,<NA>,6472960,BMG507641022,@JAT,HKG,6719.0,100.0,0.0,0.0,0.0,1.0,2020-10-20
2,19.0,PRIMERICA CORP,PRIMERICA,39733.0,741587109,0703354,US7415871091,AC,USA,6331.0,78.0,0.0,0.0,1.0,1.0,2020-10-20
3,23.0,BECOR WESTERN INC,BECOR WESTERN INC.,30530.0,075873109,<NA>,US0758731097,BCW,USA,3531.0,97.0,0.0,0.0,1.0,1.0,2020-10-20
4,24.0,AMERICAN AIRLINES INC,AMERICAN AIRLINES GROUP INC,121791.0,02376R102,BCV7KT2,US02376R1023,AMR,USA,4512.0,88.0,1.0,0.0,1.0,1.0,2020-10-20
5,26.0,DANAHER CORP,DANAHER CORPORATION,32737.0,235851102,2250870,US2358511028,DMG,USA,3823.0,77.0,1.0,1.0,1.0,1.0,2020-10-20
6,27.0,STANDARD BRANDS PAINT CO,STANDARD BRANDS PAINT COMPANY,41411.0,853156206,B0323L4,US8531562069,<NA>,USA,2851.0,91.0,1.0,0.0,1.0,1.0,2020-10-20
7,28.0,SEAMAN FURNITURE CO,SEAMAN FURNITURE CO,<NA>,812163301,<NA>,US8121633015,SMNF,USA,5712.0,100.0,1.0,1.0,1.0,1.0,2020-10-20
8,29.0,ORION CAPITAL CORP,ORION CAPITAL CORP,<NA>,686268103,2662497,US6862681037,OC,USA,6351.0,100.0,0.0,1.0,1.0,1.0,2020-10-20
9,36.0,GREAT WESTERN BANK,GREAT WESTERN BANCORP INC,124111.0,391416104,BRHZ1X6,US3914161043,GWB,USA,6022.0,79.0,1.0,0.0,0.0,1.0,2020-10-20


In [45]:
display(db.query("""
SELECT d.companyid AS dealscan_companyid,
       d.company AS dealscan_company,
       d.cusip AS dealscan_cusip,
       d.isin,
       d.ibtic,
       c.companyid AS ciq_companyid,
       c.companyname AS ciq_companyname,
       g.gvkey
FROM wrdsapps_link_dealscan_wscope.dswslink d
LEFT JOIN ciq.wrds_cusip c
  ON d.cusip = c.cusip
LEFT JOIN ciq.wrds_gvkey g
  ON c.companyid = g.companyid
WHERE d.iso3 = 'USA'
  AND d.cusip IS NOT NULL
LIMIT 50
"""))

,dealscan_companyid,dealscan_company,dealscan_cusip,isin,ibtic,ciq_companyid,ciq_companyname,gvkey
0,4.0,AMERICAN COMMUNITY DEVELOPMENT GROUP,02520G101,<NA>,CHZM,47937481.0,American Community Development Group Inc.,006300
1,4.0,AMERICAN COMMUNITY DEVELOPMENT GROUP,02520G101,<NA>,CHZM,47937481.0,American Community Development Group Inc.,126096
2,19.0,PRIMERICA CORP,741587109,US7415871091,AC,869527.0,"Primerica, Inc.",001414
3,23.0,BECOR WESTERN INC,075873109,US0758731097,BCW,25864.0,Caterpillar Global Mining LLC,024363
4,24.0,AMERICAN AIRLINES INC,02376R102,US02376R1023,AMR,168569.0,American Airlines Group Inc.,001045
5,26.0,DANAHER CORP,235851102,US2358511028,DMG,265621.0,Danaher Corporation,003735
6,27.0,STANDARD BRANDS PAINT CO,853156206,US8531562069,<NA>,34732.0,Standard Brands Paint Co.,009985
7,28.0,SEAMAN FURNITURE CO,812163301,US8121633015,SMNF,34139.0,"Seaman Furniture Company, Inc.",009557
8,29.0,ORION CAPITAL CORP,686268103,US6862681037,OC,294130.0,Orion Capital Corporation,008186
9,36.0,GREAT WESTERN BANK,391416104,US3914161043,GWB,46275772.0,"Great Western Bancorp, Inc.",021616


In [46]:
display(db.query("""
SELECT COUNT(DISTINCT d.companyid) AS dealscan_companies,
       COUNT(DISTINCT CASE WHEN c.companyid IS NOT NULL THEN d.companyid END) AS linked_to_ciq,
       COUNT(DISTINCT CASE WHEN g.gvkey IS NOT NULL THEN d.companyid END) AS linked_to_gvkey
FROM wrdsapps_link_dealscan_wscope.dswslink d
LEFT JOIN ciq.wrds_cusip c
  ON d.cusip = c.cusip
LEFT JOIN ciq.wrds_gvkey g
  ON c.companyid = g.companyid
WHERE d.iso3 = 'USA'
"""))

,dealscan_companies,linked_to_ciq,linked_to_gvkey
0,8875,8433,8367


In [48]:
display(db.query("""
SELECT
    COUNT(DISTINCT f.facilityid) AS facilities,
    COUNT(DISTINCT f.borrowercompanyid) AS companies
FROM dealscan.facility f
JOIN wrdsapps_link_dealscan_wscope.dswslink d
  ON f.borrowercompanyid = d.companyid
"""))

,facilities,companies
0,125475,17354


In [50]:
db._db.connection.rollback()

display(db.query("""
SELECT
    COUNT(DISTINCT g.gvkey) AS gvkeys,
    COUNT(DISTINCT d.companyid) AS dealscan_companies,
    COUNT(DISTINCT c.companyid) AS ciq_companies
FROM wrdsapps_link_dealscan_wscope.dswslink d
LEFT JOIN ciq.wrds_cusip c
  ON d.cusip = c.cusip
LEFT JOIN ciq.wrds_gvkey g
  ON c.companyid = g.companyid
WHERE d.iso3 = 'USA'
  AND d.cusip IS NOT NULL
"""))

,gvkeys,dealscan_companies,ciq_companies
0,8281,8759,8334


In [52]:
display(db.query("""
WITH linked_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    LEFT JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    LEFT JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    COUNT(DISTINCT l.gvkey) AS linked_gvkeys,
    COUNT(DISTINCT f.gvkey) AS gvkeys_in_compustat
FROM linked_gvkeys l
LEFT JOIN comp.fundq f
    ON l.gvkey = f.gvkey
"""))

,linked_gvkeys,gvkeys_in_compustat
0,9154,9033


In [53]:
display(db.query("""
WITH linked_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    LEFT JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    LEFT JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    COUNT(DISTINCT gvkey) AS gvkeys,
    AVG(n_quarters) AS avg_quarters,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY n_quarters) AS median_quarters,
    MIN(n_quarters) AS min_quarters,
    MAX(n_quarters) AS max_quarters
FROM (
    SELECT
        f.gvkey,
        COUNT(*) AS n_quarters
    FROM comp.fundq f
    JOIN linked_gvkeys l
      ON f.gvkey = l.gvkey
    GROUP BY f.gvkey
) x
"""))

,gvkeys,avg_quarters,median_quarters,min_quarters,max_quarters
0,9033,78.438835,62.0,3,261


In [55]:
display(db.query("""
WITH linked_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    LEFT JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    LEFT JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    k.eventtype,
    COUNT(*) AS n_events,
    COUNT(DISTINCT k.gvkey) AS n_event_gvkeys,
    COUNT(DISTINCT l.gvkey) AS n_linked_event_gvkeys
FROM ciq.wrds_keydev k
JOIN linked_gvkeys l
  ON k.gvkey = l.gvkey
WHERE k.eventtype IN (
    'Debt Defaults',
    'Bankruptcy - Filing'
)
  AND k.announcedate BETWEEN '2000-01-01' AND '2024-12-31'
GROUP BY k.eventtype
ORDER BY n_events DESC
"""))

,eventtype,n_events,n_event_gvkeys,n_linked_event_gvkeys
0,Bankruptcy - Filing,1808,1223,1223
1,Debt Defaults,188,144,144


In [56]:
display(db.query("""
WITH linked_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    LEFT JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    LEFT JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    EXTRACT(YEAR FROM k.announcedate) AS year,
    COUNT(*) AS events,
    COUNT(DISTINCT k.gvkey) AS firms
FROM ciq.wrds_keydev k
JOIN linked_gvkeys l
  ON k.gvkey = l.gvkey
WHERE k.eventtype IN (
    'Bankruptcy - Filing',
    'Debt Defaults'
)
  AND k.announcedate BETWEEN '2000-01-01' AND '2024-12-31'
GROUP BY 1
ORDER BY 1
"""))

,year,events,firms
0,2000.0,55,55
1,2001.0,145,130
2,2002.0,147,119
3,2003.0,116,99
4,2004.0,80,61
5,2005.0,61,47
6,2006.0,46,40
7,2007.0,72,54
8,2008.0,89,76
9,2009.0,191,149


In [57]:
display(db.query("""
SELECT
    MIN(f.facilitystartdate) AS first_loan,
    MAX(f.facilitystartdate) AS last_loan,
    MIN(q.datadate) AS first_financial,
    MAX(q.datadate) AS last_financial
FROM dealscan.facility f
JOIN wrdsapps_link_dealscan_wscope.dswslink d
  ON f.borrowercompanyid = d.companyid
JOIN ciq.wrds_cusip c
  ON d.cusip = c.cusip
JOIN ciq.wrds_gvkey g
  ON c.companyid = g.companyid
JOIN comp.fundq q
  ON g.gvkey = q.gvkey
"""))

,first_loan,last_loan,first_financial,last_financial
0,1982-04-20,2020-09-10,1961-03-31,2026-05-31


In [58]:
display(db.query("""
WITH borrowers AS (
    SELECT DISTINCT g.gvkey
    FROM dealscan.facility f
    JOIN wrdsapps_link_dealscan_wscope.dswslink d
      ON f.borrowercompanyid = d.companyid
    JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
)

SELECT
    COUNT(*) AS borrower_quarters,
    COUNT(DISTINCT q.gvkey) AS gvkeys
FROM comp.fundq q
JOIN borrowers b
  ON q.gvkey = b.gvkey
WHERE q.datadate BETWEEN '2000-01-01' AND '2024-12-31'
"""))


,borrower_quarters,gvkeys
0,344171,6973


In [60]:
db._db.connection.rollback()

display(db.query("""
SELECT
    gvkey,
    companyid,
    companyname,
    eventtype,
    announcedate,
    headline
FROM ciq.wrds_keydev
WHERE eventtype IN ('Bankruptcy - Filing', 'Debt Defaults')
  AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
  AND gvkey IS NOT NULL
LIMIT 10
"""))

,gvkey,companyid,companyname,eventtype,announcedate,headline
0,235716,18507.0,2M Invest A/S,Bankruptcy - Filing,2002-07-19,Bankruptcy related news reported for 2M Invest...
1,126136,19609.0,"Charter Communications, Inc.",Bankruptcy - Filing,2009-03-27,Charter Communications Inc. Filed for Bankruptcy
2,025061,20371.0,The FINOVA Group Inc.,Bankruptcy - Filing,2001-03-07,"The FINOVA Group, Inc. Filed for Bankruptcy"
3,063755,21379.0,"Kitty Hawk, Inc.",Bankruptcy - Filing,2000-05-01,Kitty Hawk Inc. has filed for bankruptcy.
4,063755,21379.0,"Kitty Hawk, Inc.",Bankruptcy - Filing,2007-10-15,"Kitty Hawk, Inc. Filed for Bankruptcy"
5,065785,21507.0,"LINC Capital, Inc.",Bankruptcy - Filing,2001-02-01,Involuntary Petition Filed Against LINC Capita...
6,006096,21609.0,Mallinckrodt LLC,Bankruptcy - Filing,2020-10-12,Mallinckrodt plc Filed for Bankruptcy
7,006096,21609.0,Mallinckrodt LLC,Bankruptcy - Filing,2020-10-12,Mallinckrodt plc Filed for Bankruptcy
8,006096,21609.0,Mallinckrodt LLC,Bankruptcy - Filing,2023-08-28,Mallinckrodt plc Filed for Bankruptcy
9,061481,22409.0,"Phar-Mor, Inc.",Bankruptcy - Filing,2001-09-24,"Phar Mor, Inc. Filed for Bankruptcy"


In [61]:
display(db.query("""
WITH borrowers AS (
    SELECT DISTINCT g.gvkey
    FROM dealscan.facility f
    JOIN wrdsapps_link_dealscan_wscope.dswslink d
      ON f.borrowercompanyid = d.companyid
    JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
),

events AS (
    SELECT DISTINCT gvkey
    FROM ciq.wrds_keydev
    WHERE eventtype IN (
        'Bankruptcy - Filing',
        'Debt Defaults'
    )
      AND gvkey IS NOT NULL
)

SELECT
    COUNT(DISTINCT b.gvkey) AS borrower_gvkeys,
    COUNT(DISTINCT e.gvkey) AS event_gvkeys,
    COUNT(DISTINCT CASE WHEN e.gvkey IS NOT NULL THEN b.gvkey END)
        AS borrower_event_overlap
FROM borrowers b
LEFT JOIN events e
    ON b.gvkey = e.gvkey
"""))

,borrower_gvkeys,event_gvkeys,borrower_event_overlap
0,9154,1411,1411


In [66]:
display(db.query("""
SELECT DISTINCT eventtype
FROM ciq.wrds_keydev
WHERE POSITION('bankrupt' IN LOWER(eventtype)) > 0
   OR POSITION('default' IN LOWER(eventtype)) > 0
ORDER BY eventtype
"""))

,eventtype
0,Bankruptcy – Asset Sale/Liquidation
1,Bankruptcy - Conclusion
2,Bankruptcy - Emergence/Exit
3,Bankruptcy - Filing
4,Bankruptcy – Financing
5,Bankruptcy - Other
6,Bankruptcy – Reorganization
7,Debt Defaults


In [67]:
display(db.query("""
WITH borrowers AS (
    SELECT DISTINCT g.gvkey
    FROM dealscan.facility f
    JOIN wrdsapps_link_dealscan_wscope.dswslink d
      ON f.borrowercompanyid = d.companyid
    JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
),

panel AS (
    SELECT q.gvkey, q.datadate
    FROM comp.fundq q
    JOIN borrowers b
      ON q.gvkey = b.gvkey
    WHERE q.datadate BETWEEN '2000-01-01' AND '2024-12-31'
),

events AS (
    SELECT DISTINCT gvkey, announcedate::date AS event_date
    FROM ciq.wrds_keydev
    WHERE eventtype IN ('Bankruptcy - Filing', 'Debt Defaults')
      AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
      AND gvkey IS NOT NULL
)

SELECT
    COUNT(*) AS total_borrower_quarters,
    SUM(CASE WHEN e1.gvkey IS NOT NULL THEN 1 ELSE 0 END) AS default_1q_obs,
    SUM(CASE WHEN e4.gvkey IS NOT NULL THEN 1 ELSE 0 END) AS default_4q_obs,
    SUM(CASE WHEN e8.gvkey IS NOT NULL THEN 1 ELSE 0 END) AS default_8q_obs
FROM panel p
LEFT JOIN events e1
  ON p.gvkey = e1.gvkey
 AND e1.event_date > p.datadate
 AND e1.event_date <= p.datadate + INTERVAL '3 months'
LEFT JOIN events e4
  ON p.gvkey = e4.gvkey
 AND e4.event_date > p.datadate
 AND e4.event_date <= p.datadate + INTERVAL '12 months'
LEFT JOIN events e8
  ON p.gvkey = e8.gvkey
 AND e8.event_date > p.datadate
 AND e8.event_date <= p.datadate + INTERVAL '24 months'
"""))

,total_borrower_quarters,default_1q_obs,default_4q_obs,default_8q_obs
0,346987,1485,5753,9805


In [69]:
display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    COUNT(*) AS rows,
    COUNT(atq) AS assets,
    COUNT(ltq) AS liabilities,
    COUNT(cheq) AS cash,
    COUNT(dlttq) AS long_term_debt,
    COUNT(dlcq) AS current_debt,
    COUNT(oibdpq) AS ebitda,
    COUNT(saleq) AS sales,
    COUNT(prccq) AS price
FROM comp.fundq
WHERE gvkey IN (
    SELECT gvkey
    FROM borrower_gvkeys
)
"""))

,rows,assets,liabilities,cash,long_term_debt,current_debt,ebitda,sales,price
0,40640,35908,35836,35734,35867,34489,33410,39291,34506


In [72]:
display(db.query("""
SELECT
    COUNT(DISTINCT gvkey) AS gvkeys
FROM comp.company
"""))

,gvkeys
0,57841


In [74]:
display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)
SELECT
    COUNT(DISTINCT gvkey) AS gvkeys,
    COUNT(*) AS quarters
FROM comp.g_fundq
WHERE gvkey IN (
    SELECT gvkey
    FROM borrower_gvkeys
)
"""))

,gvkeys,quarters
0,106,7207


In [ ]:
display(db.query("""
WITH us AS (
    SELECT DISTINCT gvkey
    FROM comp.fundq
),
global AS (
    SELECT DISTINCT gvkey
    FROM comp.g_fundq
)
SELECT
    COUNT(*) AS overlap
FROM us u
JOIN global g
  ON u.gvkey = g.gvkey
"""))

,overlap
0,1060


In [77]:
display(db.query("""
SELECT
    publicprivate,
    COUNT(DISTINCT companyid) AS companies
FROM dealscan.company
GROUP BY publicprivate
ORDER BY companies DESC
"""))

,publicprivate,companies
0,<NA>,68032
1,Private,65172
2,Public,15114


In [78]:
display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
),

us_match AS (
    SELECT DISTINCT b.gvkey
    FROM borrower_gvkeys b
    JOIN comp.fundq f
      ON b.gvkey = f.gvkey
),

global_match AS (
    SELECT DISTINCT b.gvkey
    FROM borrower_gvkeys b
    JOIN comp.g_fundq f
      ON b.gvkey = f.gvkey
)

SELECT
    (SELECT COUNT(*) FROM borrower_gvkeys) AS borrower_gvkeys,
    (SELECT COUNT(*) FROM us_match) AS us_fundq,
    (SELECT COUNT(*) FROM global_match) AS global_fundq
"""))

,borrower_gvkeys,us_fundq,global_fundq
0,835,715,106


In [79]:
display(db.query("""
WITH event_firms AS (
    SELECT DISTINCT gvkey
    FROM ciq.wrds_keydev
    WHERE eventtype IN (
        'Bankruptcy - Filing',
        'Debt Defaults'
    )
)

SELECT
    COUNT(DISTINCT e.gvkey) AS event_firms,
    COUNT(DISTINCT f.gvkey) AS event_firms_with_fundamentals
FROM event_firms e
LEFT JOIN (
    SELECT DISTINCT gvkey
    FROM comp.fundq
    UNION
    SELECT DISTINCT gvkey
    FROM comp.g_fundq
) f
ON e.gvkey = f.gvkey
"""))

,event_firms,event_firms_with_fundamentals
0,4184,3747


In [2]:
db._db.connection.rollback()

display(db.query("""
WITH borrowers AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
),

event_firms AS (
    SELECT DISTINCT gvkey
    FROM ciq.wrds_keydev
    WHERE eventtype IN (
        'Bankruptcy - Filing',
        'Debt Defaults'
    )
      AND gvkey IS NOT NULL
)

SELECT
    COUNT(*) AS event_borrowers
FROM borrowers b
JOIN event_firms e
  ON b.gvkey = e.gvkey
"""))

,event_borrowers
0,120


In [5]:
db._db.connection.rollback()

display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_cusip c
      ON d.cusip = c.cusip
    JOIN ciq.wrds_gvkey g
      ON c.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
),

panel AS (
    SELECT
        gvkey,
        EXTRACT(YEAR FROM datadate) AS year,
        atq,
        ltq,
        cheq,
        dlttq,
        dlcq,
        oibdpq,
        saleq
    FROM comp.fundq
    WHERE gvkey IN (SELECT gvkey FROM borrower_gvkeys)

    UNION ALL

    SELECT
        gvkey,
        EXTRACT(YEAR FROM datadate) AS year,
        atq,
        ltq,
        cheq,
        dlttq,
        dlcq,
        oibdpq,
        saleq
    FROM comp.g_fundq
    WHERE gvkey IN (SELECT gvkey FROM borrower_gvkeys)
)

SELECT
    year,
    COUNT(*) AS quarters,
    ROUND(100.0 * AVG(CASE WHEN atq IS NOT NULL THEN 1 ELSE 0 END),1) AS assets_pct,
    ROUND(100.0 * AVG(CASE WHEN ltq IS NOT NULL THEN 1 ELSE 0 END),1) AS liabilities_pct,
    ROUND(100.0 * AVG(CASE WHEN cheq IS NOT NULL THEN 1 ELSE 0 END),1) AS cash_pct,
    ROUND(100.0 * AVG(CASE WHEN dlttq IS NOT NULL THEN 1 ELSE 0 END),1) AS ltd_pct,
    ROUND(100.0 * AVG(CASE WHEN dlcq IS NOT NULL THEN 1 ELSE 0 END),1) AS std_pct,
    ROUND(100.0 * AVG(CASE WHEN oibdpq IS NOT NULL THEN 1 ELSE 0 END),1) AS ebitda_pct,
    ROUND(100.0 * AVG(CASE WHEN saleq IS NOT NULL THEN 1 ELSE 0 END),1) AS sales_pct
FROM panel
WHERE year BETWEEN 2000 AND 2024
GROUP BY year
ORDER BY year
"""))

,year,quarters,assets_pct,liabilities_pct,cash_pct,ltd_pct,std_pct,ebitda_pct,sales_pct
0,2000.0,20016,96.4,96.3,95.7,95.7,89.0,86.4,98.1
1,2001.0,18790,96.5,96.5,96.3,95.9,89.7,86.5,97.9
2,2002.0,18212,95.4,95.4,95.3,94.8,88.9,86.9,97.2
3,2003.0,17855,94.0,94.0,93.9,93.5,87.8,86.9,97.2
4,2004.0,17437,94.5,94.4,94.4,93.9,88.3,87.3,97.2
5,2005.0,17175,94.2,94.1,94.1,93.5,88.2,88.0,96.8
6,2006.0,16699,94.9,94.7,94.8,94.1,88.9,89.2,97.5
7,2007.0,15795,95.9,95.7,95.8,95.2,90.0,89.7,97.2
8,2008.0,15204,95.5,95.4,95.4,94.6,89.6,89.8,97.0
9,2009.0,14760,94.7,94.4,94.6,93.8,88.1,90.1,96.9


In [7]:
borrower_panel = db.query("""
WITH borrowers AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
)

SELECT
    gvkey,
    datadate,
    atq,
    ltq,
    cheq,
    dlttq,
    dlcq,
    oibdpq,
    saleq
FROM comp.fundq
WHERE gvkey IN (SELECT gvkey FROM borrowers)

UNION ALL

SELECT
    gvkey,
    datadate,
    atq,
    ltq,
    cheq,
    dlttq,
    dlcq,
    oibdpq,
    saleq
FROM comp.g_fundq
WHERE gvkey IN (SELECT gvkey FROM borrowers)
""")

In [8]:
events = db.query("""
SELECT DISTINCT
    gvkey,
    announcedate::date AS event_date,
    eventtype
FROM ciq.wrds_keydev
WHERE eventtype IN ('Bankruptcy - Filing', 'Debt Defaults')
  AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
  AND gvkey IS NOT NULL
""")

In [9]:
import pandas as pd

bp = borrower_panel.copy()
bp["datadate"] = pd.to_datetime(bp["datadate"])
events["event_date"] = pd.to_datetime(events["event_date"])

tmp = bp.merge(events[["gvkey", "event_date"]], on="gvkey", how="left")
tmp["default_1q"] = (
    (tmp["event_date"] > tmp["datadate"]) &
    (tmp["event_date"] <= tmp["datadate"] + pd.DateOffset(months=3))
)

dist_1q = (
    tmp[tmp["default_1q"]]
    .groupby("gvkey")
    .size()
    .value_counts()
    .sort_index()
    .reset_index()
)

dist_1q.columns = ["default_quarters", "firms"]
display(dist_1q)

,default_quarters,firms
0,1,42
1,2,3
2,5,1


In [11]:
display(pd.DataFrame({
    "event_gvkeys": [events["gvkey"].nunique()],
    "borrower_panel_gvkeys": [borrower_panel["gvkey"].nunique()],
    "overlap": [
        len(
            set(events["gvkey"].astype(str))
            & set(borrower_panel["gvkey"].astype(str))
        )
    ]
}))

,event_gvkeys,borrower_panel_gvkeys,overlap
0,3811,804,112


In [12]:
tmp = borrower_panel.merge(
    events[["gvkey", "event_date"]],
    on="gvkey",
    how="inner"
)

firm_event_summary = (
    tmp.groupby("gvkey")
       .agg(
           first_financial=("datadate", "min"),
           last_financial=("datadate", "max"),
           first_event=("event_date", "min"),
           last_event=("event_date", "max")
       )
)

display(
    pd.DataFrame({
        "overlap_firms": [len(firm_event_summary)],
        "event_after_first_financial": [
            (
                firm_event_summary["last_event"]
                >= firm_event_summary["first_financial"]
            ).sum()
        ],
        "event_during_financial_history": [
            (
                (firm_event_summary["last_event"]
                 >= firm_event_summary["first_financial"])
                &
                (firm_event_summary["first_event"]
                 <= firm_event_summary["last_financial"])
            ).sum()
        ]
    })
)

,overlap_firms,event_after_first_financial,event_during_financial_history
0,112,111,31


In [13]:
display(pd.DataFrame({
    "borrower_firms": [borrower_panel["gvkey"].nunique()],
    "event_firms": [events["gvkey"].nunique()],
    "overlap_firms": [
        len(set(borrower_panel["gvkey"]) & set(events["gvkey"]))
    ],
    "event_during_financial_history": [31]
}))

,borrower_firms,event_firms,overlap_firms,event_during_financial_history
0,804,3811,112,31


In [14]:
display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
),

events AS (
    SELECT
        gvkey,
        announcedate::date AS event_date
    FROM ciq.wrds_keydev
    WHERE eventtype IN (
        'Bankruptcy - Filing',
        'Debt Defaults'
    )
)

SELECT
    EXTRACT(YEAR FROM event_date) AS year,
    COUNT(DISTINCT gvkey) AS firms
FROM events
WHERE gvkey IN (SELECT gvkey FROM borrower_gvkeys)
GROUP BY year
ORDER BY year
"""))

,year,firms
0,1998.0,1
1,1999.0,3
2,2000.0,3
3,2001.0,21
4,2002.0,12
5,2003.0,8
6,2004.0,7
7,2005.0,4
8,2006.0,4
9,2007.0,2


In [15]:
display(db.query("""
WITH borrower_gvkeys AS (
    SELECT DISTINCT g.gvkey
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    WHERE g.gvkey IS NOT NULL
),

events AS (
    SELECT DISTINCT
        gvkey,
        announcedate::date AS event_date
    FROM ciq.wrds_keydev
    WHERE eventtype IN (
        'Bankruptcy - Filing',
        'Debt Defaults'
    )
      AND gvkey IS NOT NULL
),

fundamentals AS (
    SELECT gvkey,
           MIN(datadate) AS first_fin,
           MAX(datadate) AS last_fin
    FROM (
        SELECT gvkey, datadate FROM comp.fundq
        UNION ALL
        SELECT gvkey, datadate FROM comp.g_fundq
    ) x
    GROUP BY gvkey
)

SELECT
    COUNT(DISTINCT e.gvkey) AS event_firms,
    COUNT(DISTINCT CASE
        WHEN e.event_date BETWEEN f.first_fin AND f.last_fin
        THEN e.gvkey
    END) AS event_within_financial_history,
    COUNT(DISTINCT CASE
        WHEN e.event_date BETWEEN f.first_fin - INTERVAL '2 years'
                             AND f.last_fin + INTERVAL '2 years'
        THEN e.gvkey
    END) AS event_within_2yr_buffer
FROM events e
JOIN borrower_gvkeys b
  ON e.gvkey = b.gvkey
LEFT JOIN fundamentals f
  ON e.gvkey = f.gvkey
"""))

,event_firms,event_within_financial_history,event_within_2yr_buffer
0,120,32,85


In [16]:
display(db.query("""
SELECT
    eventtype,
    COUNT(DISTINCT gvkey) AS firms
FROM ciq.wrds_keydev
WHERE eventtype IN (
    'Bankruptcy - Filing',
    'Debt Defaults',
    'Bankruptcy - Conclusion',
    'Bankruptcy - Emergence/Exit',
    'Bankruptcy – Reorganization',
    'Bankruptcy – Asset Sale/Liquidation',
    'Bankruptcy – Financing',
    'Delistings'
)
GROUP BY eventtype
ORDER BY firms DESC
"""))

,eventtype,firms
0,Delistings,33762
1,Bankruptcy - Filing,3448
2,Bankruptcy – Asset Sale/Liquidation,2026
3,Bankruptcy – Reorganization,1770
4,Bankruptcy - Emergence/Exit,1702
5,Bankruptcy – Financing,1494
6,Bankruptcy - Conclusion,1414
7,Debt Defaults,901


In [19]:
display(pd.DataFrame({
    "total_quarters": [len(borrower_panel)],
    "firms": [borrower_panel.gvkey.nunique()],
    "assets_pct": [100 * borrower_panel.atq.notna().mean()],
    "liabilities_pct": [100 * borrower_panel.ltq.notna().mean()],
    "cash_pct": [100 * borrower_panel.cheq.notna().mean()],
    "ltd_pct": [100 * borrower_panel.dlttq.notna().mean()],
    "std_pct": [100 * borrower_panel.dlcq.notna().mean()],
    "ebitda_pct": [100 * borrower_panel.oibdpq.notna().mean()],
    "sales_pct": [100 * borrower_panel.saleq.notna().mean()]
}))

,total_quarters,firms,assets_pct,liabilities_pct,cash_pct,ltd_pct,std_pct,ebitda_pct,sales_pct
0,47847,804,86.534161,86.377411,86.655381,83.875687,80.928794,79.039438,91.30562


In [21]:
borrower_panel["datadate"] = pd.to_datetime(borrower_panel["datadate"])

display(
    borrower_panel.groupby(
        borrower_panel["datadate"].dt.year
    ).agg(
        firms=("gvkey", "nunique"),
        quarters=("gvkey", "size")
    ).reset_index()
)

,datadate,firms,quarters
0,1962,12,48
1,1963,13,52
2,1964,13,52
3,1965,17,59
4,1966,28,102
...,...,...,...
60,2022,144,580
61,2023,142,548
62,2024,133,522
63,2025,123,482


In [23]:
import pandas as pd

events = db.query("""
SELECT DISTINCT
    gvkey,
    announcedate::date AS event_date,
    eventtype
FROM ciq.wrds_keydev
WHERE eventtype IN ('Bankruptcy - Filing', 'Debt Defaults')
  AND announcedate BETWEEN '2000-01-01' AND '2024-12-31'
  AND gvkey IS NOT NULL
""")

borrower_panel["datadate"] = pd.to_datetime(borrower_panel["datadate"])
events["event_date"] = pd.to_datetime(events["event_date"])

bp = borrower_panel.copy()
bp["_row_id"] = range(len(bp))

tmp = bp[["_row_id", "gvkey", "datadate"]].merge(
    events[["gvkey", "event_date"]],
    on="gvkey",
    how="left"
)

tmp["default_1q"] = (
    (tmp["event_date"] > tmp["datadate"]) &
    (tmp["event_date"] <= tmp["datadate"] + pd.DateOffset(months=3))
)

tmp["default_4q"] = (
    (tmp["event_date"] > tmp["datadate"]) &
    (tmp["event_date"] <= tmp["datadate"] + pd.DateOffset(months=12))
)

tmp["default_8q"] = (
    (tmp["event_date"] > tmp["datadate"]) &
    (tmp["event_date"] <= tmp["datadate"] + pd.DateOffset(months=24))
)

labels = (
    tmp.groupby("_row_id")[["default_1q", "default_4q", "default_8q"]]
    .max()
    .astype(int)
)

borrower_panel = borrower_panel.join(labels)

display(
    borrower_panel[["default_1q", "default_4q", "default_8q"]]
    .sum()
    .to_frame("positive_quarters")
)

,positive_quarters
default_1q,52
default_4q,293
default_8q,653


In [24]:
pos = borrower_panel[borrower_panel["default_4q"] == 1]

display(pd.DataFrame({
    "rows": [len(pos)],
    "firms": [pos["gvkey"].nunique()],
    "assets_pct": [100 * pos["atq"].notna().mean()],
    "liabilities_pct": [100 * pos["ltq"].notna().mean()],
    "cash_pct": [100 * pos["cheq"].notna().mean()],
    "ltd_pct": [100 * pos["dlttq"].notna().mean()],
    "std_pct": [100 * pos["dlcq"].notna().mean()],
    "ebitda_pct": [100 * pos["oibdpq"].notna().mean()],
    "sales_pct": [100 * pos["saleq"].notna().mean()]
}))

,rows,firms,assets_pct,liabilities_pct,cash_pct,ltd_pct,std_pct,ebitda_pct,sales_pct
0,293,77,96.928328,96.928328,97.269625,95.56314,95.56314,91.467577,97.952218


In [25]:
display(
    borrower_panel.loc[borrower_panel["default_4q"] == 1]
    .groupby(borrower_panel.loc[borrower_panel["default_4q"] == 1, "datadate"].dt.year)
    .agg(
        firms=("gvkey", "nunique"),
        quarters=("gvkey", "size")
    )
    .reset_index()
)

,datadate,firms,quarters
0,1999,3,6
1,2000,22,48
2,2001,22,48
3,2002,10,22
4,2003,5,11
5,2004,3,7
6,2005,4,7
7,2006,4,8
8,2007,3,8
9,2008,14,33


In [26]:
display(
    borrower_panel.loc[borrower_panel["default_4q"] == 1]
    [["atq","ltq","cheq","dlttq","dlcq","oibdpq","saleq"]]
    .describe()
)

,atq,ltq,cheq,dlttq,dlcq,oibdpq,saleq
count,284.0,284.0,285.0,280.0,280.0,268.0,287.0
mean,2931.816204,1992.38407,107.579249,904.964143,516.803964,36.628687,359.550923
std,15773.858602,10528.688289,451.239138,4838.609324,4485.463888,389.127492,1352.310164
min,1.675,2.729,-0.066,0.0,0.0,-3173.2,0.0
25%,67.029,55.825,2.59,0.5405,2.07625,-4.9755,17.183
50%,277.054,266.6475,10.137,37.6585,9.1325,-0.571,60.324
75%,642.38525,759.30625,39.728,254.98825,105.57175,10.78825,241.421
max,156658.4,115119.8,4653.7,50490.6,71452.1,4204.2,14678.1


In [31]:
db._db.connection.rollback()

a = pd.DataFrame({
    "libraries": db._db.list_libraries()
})

print(a.to_string())

                           libraries
0                         aha_sample
1                            ahasamp
2                           auditsmp
3                       auditsmp_all
4                               bank
5                           bank_all
6                  bank_premium_samp
7                           banksamp
8                              block
9                          block_all
10                     boardex_trial
11                          boardsmp
12                 bvd_amadeus_trial
13                bvd_bvdbankf_trial
14                   bvd_orbis_trial
15                           bvdsamp
16                   calcbench_trial
17                          calcbnch
18                       candid_samp
19                              cboe
20                          cboe_all
21                       cboe_sample
22                          cboesamp
23                           cddsamp
24                               ciq
25                      ciq_capstrct
2

In [33]:
a = pd.DataFrame({

    "tables": db._db.list_tables(library="fisd")

})

print(a.to_string())

                              tables
0                         fisd_agent
1            fisd_amount_outstanding
2                  fisd_amt_out_hist
3                fisd_announced_call
4                    fisd_bankruptcy
5              fisd_bankruptcy_agent
6         fisd_bondholder_protective
7                 fisd_call_schedule
8                fisd_change_formula
9               fisd_change_schedule
10                         fisd_code
11                      fisd_contact
12                  fisd_convertible
13      fisd_convertible_addit_terms
14          fisd_convertible_history
15   fisd_convertible_issuer_history
16  fisd_convertible_soft_call_sched
17          fisd_coupon_formulaindex
18                  fisd_coupon_info
19             fisd_exchange_listing
20             fisd_foreign_currency
21                 fisd_ipo_clawback
22                        fisd_issue
23               fisd_issue_affected
24                 fisd_issue_agents
25                fisd_issue_default
2

In [34]:
display(db.query("""

SELECT *

FROM fisd.fisd_rating_hist

LIMIT 10

"""))

,issue_id,rating_type,rating_date,rating,rating_status,reason
0,1.0,MR,1993-05-26,Baa3,<NA>,IL
1,1.0,SPR,1995-02-22,BBB-,<NA>,IL
2,1.0,FR,1996-01-04,BBB-,<NA>,IL
3,1.0,MR,1997-01-23,Baa2,<NA>,UPG
4,1.0,SPR,1997-02-07,BBB-,<NA>,AFRM
5,1.0,SPR,1997-07-03,BBB,<NA>,UPG
6,1.0,FR,1997-08-20,BBB,<NA>,UPG
7,1.0,FR,1999-02-01,BBB,<NA>,AFRM
8,1.0,MR,1999-12-29,Baa2,<NA>,IR
9,1.0,FR,2000-08-16,BBB,<NA>,IR


In [35]:
display(db.query("""
SELECT *
FROM fisd.issue_issuer
LIMIT 10
"""))

,issue_id,issuer_id,prospectus_issuer_name,issuer_cusip,issue_cusip,issue_name,maturity,security_level,security_pledge,enhancement,...,complete_cusip,agent_id,cusip_name,industry_group,industry_code,esop,in_bankruptcy,parent_id,naics_code,country_domicile
0,1.0,3.0,AAR CORP,000361,AA3,NT,2001-11-01,SEN,<NA>,N,...,000361AA3,3.0,AAR CORP,1,10,N,N,3.0,336412,USA
1,2.0,3.0,AAR CORP,000361,AB1,NT,2003-10-15,SEN,<NA>,N,...,000361AB1,3.0,AAR CORP,1,10,N,N,3.0,336412,USA
2,3.0,40263.0,ABN AMRO BK N V N Y BRH,00077D,AB5,MTN,1996-01-12,SEN,<NA>,N,...,00077DAB5,5.0,ABN AMRO BK N V N Y BRH,2,20,N,N,4618.0,52211,USA
3,4.0,40263.0,ABN AMRO BK N V N Y BRH,00077D,AF6,SUB DEP NT SER B,2009-08-01,SENS,<NA>,N,...,00077DAF6,5.0,ABN AMRO BK N V N Y BRH,2,20,N,N,4618.0,52211,USA
4,5.0,40263.0,ABN AMRO BK N V N Y BRH,00077T,AA2,SUB DEP NT SER B,2023-05-15,SUB,<NA>,N,...,00077TAA2,5.0,ABN AMRO BK N V N Y BRH,2,20,N,N,4618.0,52211,USA
5,6.0,40263.0,ABN AMRO BK N V N Y BRH,00077T,AB0,SUB DEP NT SER B,2093-10-15,SUB,<NA>,N,...,00077TAB0,5.0,ABN AMRO BK N V N Y BRH,2,20,N,N,4618.0,52211,USA
6,8.0,6.0,ACF INDS INC,000800,AR3,EQUIP TR CTF SER J,2000-05-15,NON,<NA>,N,...,000800AR3,6.0,ACF INDS INC,1,10,N,N,6.0,<NA>,USA
7,9.0,6.0,ACF INDS INC,000800,AX0,EQUIP TR CTF SER L,1996-12-01,NON,<NA>,N,...,000800AX0,6.0,ACF INDS INC,1,10,N,N,6.0,<NA>,USA
8,10.0,6.0,ACF INDS INC,000800,AY8,EQUIP TR CTF SER M,1992-08-01,NON,<NA>,N,...,000800AY8,6.0,ACF INDS INC,1,10,N,N,6.0,<NA>,USA
9,11.0,6.0,ACF INDS INC,000800,BA9,SINKING FUND DEB,1996-12-15,SEN,<NA>,N,...,000800BA9,6.0,ACF INDS INC,1,10,N,N,6.0,<NA>,USA


In [36]:
display(db.query("""
SELECT *
FROM fisd.fisd_issuer
LIMIT 10
"""))

,issuer_id,agent_id,cusip_name,industry_group,industry_code,esop,in_bankruptcy,parent_id,naics_code,country_domicile
0,3.0,3.0,AAR CORP,1,10,N,N,3.0,336412,USA
1,6097.0,4.0,ABN AMRO BK N V CHICAGO BRH,2,20,N,N,43413.0,52211,USA
2,40263.0,5.0,ABN AMRO BK N V N Y BRH,2,20,N,N,4618.0,52211,USA
3,6.0,6.0,ACF INDS INC,1,10,N,N,6.0,<NA>,USA
4,7.0,7.0,ADT OPERATIONS INC,1,15,N,N,4214.0,561621,USA
5,8.0,8.0,AFG INDS INC,1,10,N,N,51895.0,<NA>,USA
6,9.0,9.0,AG SVCS AMER INC,1,15,N,N,9.0,424910,USA
7,10.0,10.0,AHSC HLDGS CORP,1,14,N,N,46333.0,424210,USA
8,11.0,11.0,AES CORP,3,30,N,N,11.0,221122,USA
9,11732.0,11.0,AES TR I,3,30,N,N,11.0,221122,USA


In [37]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

fisd_issuers AS (
    SELECT DISTINCT
        issuer_id,
        issuer_cusip,
        cusip_name
    FROM fisd.issue_issuer
    WHERE issuer_cusip IS NOT NULL
)

SELECT
    COUNT(DISTINCT b.gvkey) AS borrower_gvkeys,
    COUNT(DISTINCT f.issuer_id) AS matched_fisd_issuers,
    COUNT(DISTINCT CASE WHEN f.issuer_id IS NOT NULL THEN b.gvkey END) AS matched_borrower_gvkeys
FROM borrower_cusips b
LEFT JOIN fisd_issuers f
  ON LEFT(b.cusip, 6) = f.issuer_cusip
"""))

,borrower_gvkeys,matched_fisd_issuers,matched_borrower_gvkeys
0,787,321,260


In [38]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
),

rated_borrowers AS (
    SELECT DISTINCT
        bi.gvkey
    FROM borrower_issuers bi
    JOIN fisd.issue_issuer ii
      ON bi.issuer_id = ii.issuer_id
    JOIN fisd.fisd_rating_hist rh
      ON ii.issue_id = rh.issue_id
)

SELECT
    COUNT(DISTINCT bi.gvkey) AS matched_borrowers,
    COUNT(DISTINCT rb.gvkey) AS borrowers_with_rating_history
FROM borrower_issuers bi
LEFT JOIN rated_borrowers rb
  ON bi.gvkey = rb.gvkey
"""))

,matched_borrowers,borrowers_with_rating_history
0,260,196


In [39]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
),

ratings AS (
    SELECT
        bi.gvkey,
        rh.rating_date,
        rh.rating_type,
        rh.rating,
        rh.reason
    FROM borrower_issuers bi
    JOIN fisd.issue_issuer ii
      ON bi.issuer_id = ii.issuer_id
    JOIN fisd.fisd_rating_hist rh
      ON ii.issue_id = rh.issue_id
)

SELECT
    COUNT(*) AS rating_events,
    COUNT(DISTINCT gvkey) AS rated_borrowers,
    MIN(rating_date) AS first_rating,
    MAX(rating_date) AS last_rating
FROM ratings
"""))

,rating_events,rated_borrowers,first_rating,last_rating
0,19655,196,1985-07-08,2025-05-16


In [40]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
)

SELECT
    reason,
    COUNT(*) AS events
FROM borrower_issuers bi
JOIN fisd.issue_issuer ii
    ON bi.issuer_id = ii.issuer_id
JOIN fisd.fisd_rating_hist rh
    ON ii.issue_id = rh.issue_id
GROUP BY reason
ORDER BY events DESC
LIMIT 25
"""))

,reason,events
0,AFRM,8792
1,IN,3999
2,DNG,2950
3,IR,1727
4,UPG,1712
5,IL,190
6,MA,128
7,CONF,75
8,<NA>,57
9,CP,22


In [41]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
)

SELECT
    COUNT(DISTINCT gvkey) AS rated_borrowers,
    COUNT(DISTINCT CASE WHEN dng_count > 0 THEN gvkey END) AS borrowers_with_downgrade
FROM (
    SELECT
        bi.gvkey,
        SUM(CASE WHEN rh.reason = 'DNG' THEN 1 ELSE 0 END) AS dng_count
    FROM borrower_issuers bi
    JOIN fisd.issue_issuer ii
      ON bi.issuer_id = ii.issuer_id
    JOIN fisd.fisd_rating_hist rh
      ON ii.issue_id = rh.issue_id
    GROUP BY bi.gvkey
) x
"""))

,rated_borrowers,borrowers_with_downgrade
0,196,143


In [42]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
)

SELECT
    rating,
    COUNT(*) AS events
FROM borrower_issuers bi
JOIN fisd.issue_issuer ii
    ON bi.issuer_id = ii.issuer_id
JOIN fisd.fisd_rating_hist rh
    ON ii.issue_id = rh.issue_id
GROUP BY rating
ORDER BY events DESC
LIMIT 50
"""))

,rating,events
0,A,2319
1,A-,2038
2,A+,1775
3,BBB+,1295
4,A3,1102
5,BBB,1008
6,AA-,929
7,BBB-,859
8,A2,642
9,A1,639


In [43]:
display(db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
),

rating_firms AS (
    SELECT DISTINCT
        bi.gvkey
    FROM borrower_issuers bi
    JOIN fisd.issue_issuer ii
      ON bi.issuer_id = ii.issuer_id
    JOIN fisd.fisd_rating_hist rh
      ON ii.issue_id = rh.issue_id
)

SELECT
    COUNT(DISTINCT bp.gvkey) AS borrower_firms,
    COUNT(DISTINCT rf.gvkey) AS rated_firms,
    COUNT(DISTINCT CASE
        WHEN rf.gvkey IS NOT NULL THEN bp.gvkey
    END) AS overlap
FROM (
    SELECT DISTINCT gvkey
    FROM comp.fundq
    UNION
    SELECT DISTINCT gvkey
    FROM comp.g_fundq
) bp
LEFT JOIN rating_firms rf
    ON bp.gvkey = rf.gvkey
WHERE bp.gvkey IN (
    SELECT DISTINCT gvkey
    FROM borrower_issuers
)
"""))

,borrower_firms,rated_firms,overlap
0,258,194,194


In [45]:
ratings = db.query("""
WITH borrower_cusips AS (
    SELECT DISTINCT
        g.gvkey,
        c.cusip
    FROM wrdsapps_link_dealscan_wscope.dswslink d
    JOIN ciq.wrds_gvkey g
      ON d.companyid = g.companyid
    JOIN ciq.wrds_cusip c
      ON g.companyid = c.companyid
    WHERE g.gvkey IS NOT NULL
      AND c.cusip IS NOT NULL
),

borrower_issuers AS (
    SELECT DISTINCT
        b.gvkey,
        f.issuer_id
    FROM borrower_cusips b
    JOIN fisd.issue_issuer f
      ON LEFT(b.cusip, 6) = f.issuer_cusip
)

SELECT
    bi.gvkey,
    rh.issue_id,
    rh.rating_date,
    rh.rating,
    rh.rating_type,
    rh.reason
FROM borrower_issuers bi
JOIN fisd.issue_issuer ii
  ON bi.issuer_id = ii.issuer_id
JOIN fisd.fisd_rating_hist rh
  ON ii.issue_id = rh.issue_id
WHERE rh.rating IS NOT NULL
ORDER BY bi.gvkey, rh.rating_date
""")

display(pd.DataFrame({
    "rating_events": [len(ratings)],
    "rated_borrowers": [ratings["gvkey"].nunique()],
    "first_rating": [ratings["rating_date"].min()],
    "last_rating": [ratings["rating_date"].max()]
}))

,rating_events,rated_borrowers,first_rating,last_rating
0,19655,196,1985-07-08,2025-05-16


In [47]:
ratings["rating_date"] = pd.to_datetime(ratings["rating_date"])
borrower_panel["datadate"] = pd.to_datetime(borrower_panel["datadate"])

bp = borrower_panel.dropna(subset=["gvkey", "datadate"]).copy()
rt = ratings.dropna(subset=["gvkey", "rating_date"]).copy()

bp["gvkey"] = bp["gvkey"].astype(str)
rt["gvkey"] = rt["gvkey"].astype(str)

bp = bp.sort_values(["datadate", "gvkey"]).reset_index(drop=True)
rt = rt.sort_values(["rating_date", "gvkey"]).reset_index(drop=True)

rating_panel = pd.merge_asof(
    bp,
    rt,
    left_on="datadate",
    right_on="rating_date",
    by="gvkey",
    direction="backward"
)

display(pd.DataFrame({
    "borrower_quarters": [len(bp)],
    "quarters_with_rating": [rating_panel["rating"].notna().sum()],
    "rated_firms": [rating_panel.loc[rating_panel["rating"].notna(), "gvkey"].nunique()],
    "first_panel_date": [rating_panel["datadate"].min()],
    "last_panel_date": [rating_panel["datadate"].max()]
}))

,borrower_quarters,quarters_with_rating,rated_firms,first_panel_date,last_panel_date
0,47847,8456,183,1962-03-31,2026-05-31


In [48]:
dng = ratings.loc[ratings["reason"] == "DNG", ["gvkey", "rating_date"]].copy()
dng["gvkey"] = dng["gvkey"].astype(str)
dng["rating_date"] = pd.to_datetime(dng["rating_date"])

rp = rating_panel.copy()
rp["_row_id"] = range(len(rp))

tmp = rp[["_row_id", "gvkey", "datadate"]].merge(
    dng.rename(columns={"rating_date": "downgrade_date"}),
    on="gvkey",
    how="left"
)

tmp["downgrade_4q"] = (
    (tmp["downgrade_date"] > tmp["datadate"]) &
    (tmp["downgrade_date"] <= tmp["datadate"] + pd.DateOffset(months=12))
)

tmp["downgrade_8q"] = (
    (tmp["downgrade_date"] > tmp["datadate"]) &
    (tmp["downgrade_date"] <= tmp["datadate"] + pd.DateOffset(months=24))
)

downgrade_labels = (
    tmp.groupby("_row_id")[["downgrade_4q", "downgrade_8q"]]
    .max()
    .astype(int)
)

rating_panel = rating_panel.join(downgrade_labels)

display(
    rating_panel.loc[rating_panel["rating"].notna(), ["downgrade_4q", "downgrade_8q"]]
    .sum()
    .to_frame("positive_quarters")
)

,positive_quarters
downgrade_4q,1431
downgrade_8q,2206


In [49]:
display(
    rating_panel.loc[rating_panel["rating"].notna()]
    .groupby("rating")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="quarters")
)

,rating,quarters
0,B-,651
1,B3,546
2,B2,504
3,B+,493
4,B,473
5,CCC+,405
6,BB-,402
7,BBB-,315
8,Baa2,311
9,BBB,269


In [51]:
tmp = rating_panel.copy()

tmp["debt_assets"] = (tmp["dlttq"].fillna(0) + tmp["dlcq"].fillna(0)) / tmp["atq"]
tmp["cash_assets"] = tmp["cheq"] / tmp["atq"]
tmp["ebitda_assets"] = tmp["oibdpq"] / tmp["atq"]

downgraders = tmp[tmp["downgrade_4q"] == 1]

display(pd.DataFrame({
    "metric": [
        "debt_assets",
        "cash_assets",
        "ebitda_assets"
    ],
    "downgrade_median": [
        downgraders["debt_assets"].median(),
        downgraders["cash_assets"].median(),
        downgraders["ebitda_assets"].median()
    ],
    "all_median": [
        tmp["debt_assets"].median(),
        tmp["cash_assets"].median(),
        tmp["ebitda_assets"].median()
    ]
}))

,metric,downgrade_median,all_median
0,debt_assets,0.401671,0.179887
1,cash_assets,0.039802,0.136932
2,ebitda_assets,0.026626,0.024402


In [52]:
tmp = rating_panel.copy()

tmp["debt_assets"] = (tmp["dlttq"].fillna(0) + tmp["dlcq"].fillna(0)) / tmp["atq"]
tmp["cash_assets"] = tmp["cheq"] / tmp["atq"]
tmp["ebitda_assets"] = tmp["oibdpq"] / tmp["atq"]

downgraders = tmp[tmp["downgrade_4q"] == 1]

display(pd.DataFrame({
    "metric": [
        "debt_assets",
        "cash_assets",
        "ebitda_assets"
    ],
    "downgrade_median": [
        downgraders["debt_assets"].median(),
        downgraders["cash_assets"].median(),
        downgraders["ebitda_assets"].median()
    ],
    "all_median": [
        tmp["debt_assets"].median(),
        tmp["cash_assets"].median(),
        tmp["ebitda_assets"].median()
    ]
}))

,metric,downgrade_median,all_median
0,debt_assets,0.401671,0.179887
1,cash_assets,0.039802,0.136932
2,ebitda_assets,0.026626,0.024402
